## 📋Cenário: Análise de Vendas por Representante Comercial
Você trabalha como analista de dados em uma distribuidora. O time comercial quer entender a performance dos representantes de vendas ao longo do tempo, comparando resultados dentro de regiões e identificando tendências.

## 🎯 Problema
Com base nos dados de vendas, você precisa criar uma query que retorne:

- Para cada venda, o valor acumulado do representante no mês (running total)
- A média móvel dos últimos 3 pedidos de cada representante (por data)
- O ranking do representante dentro da sua região por valor total vendido no mês
- A diferença entre a venda atual e a venda anterior do mesmo representante (LAG)
- O percentual que cada venda representa sobre o total do representante naquele mês

In [0]:
%sql
-- DDL
CREATE OR REPLACE TABLE sandbox.projetos.vendas (
  id_venda      INT,
  id_rep        INT,
  nome_rep      STRING,
  regiao        STRING,
  data_venda    DATE,
  valor         DECIMAL(10, 2)
);

-- DML
INSERT INTO sandbox.projetos.vendas VALUES
(1,  101, 'Ana Lima',     'Sul',     '2024-01-05', 1200.00),
(2,  101, 'Ana Lima',     'Sul',     '2024-01-12', 850.00),
(3,  101, 'Ana Lima',     'Sul',     '2024-01-20', 2100.00),
(4,  101, 'Ana Lima',     'Sul',     '2024-02-03', 1750.00),
(5,  101, 'Ana Lima',     'Sul',     '2024-02-18', 960.00),
(6,  102, 'Bruno Melo',   'Sul',     '2024-01-07', 3000.00),
(7,  102, 'Bruno Melo',   'Sul',     '2024-01-15', 1400.00),
(8,  102, 'Bruno Melo',   'Sul',     '2024-01-28', 2200.00),
(9,  102, 'Bruno Melo',   'Sul',     '2024-02-10', 500.00),
(10, 102, 'Bruno Melo',   'Sul',     '2024-02-25', 1800.00),
(11, 103, 'Carla Souza',  'Norte',   '2024-01-03', 900.00),
(12, 103, 'Carla Souza',  'Norte',   '2024-01-19', 1600.00),
(13, 103, 'Carla Souza',  'Norte',   '2024-01-30', 750.00),
(14, 103, 'Carla Souza',  'Norte',   '2024-02-14', 2400.00),
(15, 103, 'Carla Souza',  'Norte',   '2024-02-27', 1100.00),
(16, 104, 'Diego Ramos',  'Norte',   '2024-01-10', 1950.00),
(17, 104, 'Diego Ramos',  'Norte',   '2024-01-22', 3200.00),
(18, 104, 'Diego Ramos',  'Norte',   '2024-02-05', 870.00),
(19, 104, 'Diego Ramos',  'Norte',   '2024-02-19', 1430.00),
(20, 104, 'Diego Ramos',  'Norte',   '2024-02-28', 2650.00);

In [0]:
%sql
with v as (

select
    id_venda,
    id_rep,
    nome_rep,
    regiao,
    data_venda,
    valor,
    extract(month from data_venda)                                   as  mes,
    sum(valor) over(
        partition by id_rep, regiao, extract(month from data_venda)) as vlr_venda_mes,
    round(avg(valor) over (
        partition by id_rep 
        order by data_venda 
        rows between 2 preceding and current row)
        ,2)                                                         as media_movel_3_pedidos,
    valor - lag(valor,1) over (
        partition by id_rep 
        order by data_venda)                                        as diff_vlr_atual_anterior
from sandbox.projetos.vendas

)
select 
    * ,
    dense_rank() over(
        partition by regiao, mes 
        order by vlr_venda_mes desc ) d_rank_vendas_regiao,
    round(valor / vlr_venda_mes *100,2)                             as pct_venda
from v
